In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Dataset

In [8]:
df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_sports.csv")
df['review_body'] = df['review_body'].str.replace('[^a-zA-ZñÑáéíóú .,]', '', regex=True)
df['review_body'] = df['review_body'].str.lower()
df = df[df.stars != 3]
df['good_product'] = (df.stars > 3).astype(int)
df.sample(5)

,stars,review_body,review_title,product_category,good_product
13026,5,todo perfecto gracias,camaras de ruedas,sports,1
4080,2,"la app es un autentico desastre, mal traducida...",No merece la pena,sports,0
12864,5,creo que en el título ya lo he dicho todo.lis ...,Muy bonitos y buena calidad,sports,1
3817,2,a fecha posterior a la de entrega del producto...,Desencantado,sports,0
9696,4,si sales a la aventura hay que reconocer que e...,super recomendable,sports,1


# Text Classification

In [10]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

In [11]:
X = df.review_body.values
y = df.good_product

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=33)

In [13]:
print(len(X_train))
print(len(X_test))

8288
2073


In [14]:
vocab_size = 5000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token='<OOV>'
)
tokenizer.fit_on_texts(X_train)

tokenized_train = tokenizer.texts_to_sequences(X_train)
tokenized_test = tokenizer.texts_to_sequences(X_test)

max_length = 50

padded_train = pad_sequences(tokenized_train, maxlen=max_length, truncating='post')
padded_test = pad_sequences(tokenized_test, maxlen=max_length, truncating='post')

# Models

In [24]:
import tensorflow.keras as keras
import numpy as np

from keras import Sequential
from keras.layers import *

In [28]:
def model_compile(model):
    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics='accuracy'
    )
    print(model.summary())

In [30]:
def model_fit(model):
    n_epochs = 20
    batch_size = 100
    model.fit(
        padded_train, y_train,
        epochs=n_epochs,
        batch_size=batch_size,
        validation_data=(padded_test, y_test),
        verbose=True
    )

## Model 1: Dense

In [29]:
keras.utils.set_random_seed(812)

model = Sequential([
    Flatten(input_shape=(max_length,)),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid'),
])

model_compile(model)

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten_1 (Flatten)         (None, 50)                0         
                                                                 
 dense_3 (Dense)             (None, 128)               6528      
                                                                 
 dense_4 (Dense)             (None, 64)                8256      
                                                                 
 dense_5 (Dense)             (None, 1)                 65        
                                                                 
Total params: 14849 (58.00 KB)
Trainable params: 14849 (58.00 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [31]:
%%time

model_fit(model)

Epoch 1/20


83/83 [==============================] - 1s 6ms/step - loss: 32.6707 - accuracy: 0.5148 - val_loss: 17.6519 - val_accuracy: 0.4969
Epoch 2/20
83/83 [==============================] - 0s 3ms/step - loss: 13.3820 - accuracy: 0.5402 - val_loss: 13.2627 - val_accuracy: 0.5239
Epoch 3/20
83/83 [==============================] - 0s 3ms/step - loss: 8.6187 - accuracy: 0.5770 - val_loss: 10.9399 - val_accuracy: 0.5041
Epoch 4/20
83/83 [==============================] - 0s 3ms/step - loss: 6.1289 - accuracy: 0.5913 - val_loss: 9.9573 - val_accuracy: 0.5031
Epoch 5/20
83/83 [==============================] - 0s 3ms/step - loss: 4.6357 - accuracy: 0.6192 - val_loss: 9.2269 - val_accuracy: 0.5176
Epoch 6/20
83/83 [==============================] - 0s 3ms/step - loss: 3.5993 - accuracy: 0.6359 - val_loss: 8.6498 - val_accuracy: 0.5176
Epoch 7/20
83/83 [==============================] - 0s 3ms/step - loss: 2.8446 - accuracy: 0.6578 - val_loss: 7.9590 - val_accuracy: 0.4954
Epoch 8/20
83

## Model 2: Dense + Embeddings

In [32]:
keras.utils.set_random_seed(812)

embd_dim = 20

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embd_dim,
        input_length=max_length
    ),
    Flatten(),
    Dense(1, activation='sigmoid'),
])

model_compile(model)

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 50, 20)            100000    
                                                                 
 flatten_2 (Flatten)         (None, 1000)              0         
                                                                 
 dense_6 (Dense)             (None, 1)                 1001      
                                                                 
Total params: 101001 (394.54 KB)
Trainable params: 101001 (394.54 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [33]:
%%time

model_fit(model)

Epoch 1/20
83/83 [==============================] - 1s 6ms/step - loss: 0.6788 - accuracy: 0.5677 - val_loss: 0.6574 - val_accuracy: 0.6425
Epoch 2/20
83/83 [==============================] - 0s 4ms/step - loss: 0.5832 - accuracy: 0.7426 - val_loss: 0.5193 - val_accuracy: 0.7704
Epoch 3/20
83/83 [==============================] - 0s 4ms/step - loss: 0.4288 - accuracy: 0.8340 - val_loss: 0.4188 - val_accuracy: 0.8157
Epoch 4/20
83/83 [==============================] - 0s 4ms/step - loss: 0.3377 - accuracy: 0.8737 - val_loss: 0.3792 - val_accuracy: 0.8292
Epoch 5/20
83/83 [==============================] - 0s 4ms/step - loss: 0.2812 - accuracy: 0.8982 - val_loss: 0.3588 - val_accuracy: 0.8398
Epoch 6/20
83/83 [==============================] - 0s 4ms/step - loss: 0.2423 - accuracy: 0.9175 - val_loss: 0.3493 - val_accuracy: 0.8437
Epoch 7/20
83/83 [==============================] - 0s 4ms/step - loss: 0.2089 - accuracy: 0.9312 - val_loss: 0.3456 - val_accuracy: 0.8442
Epoch 8/20
83/83 [==

## Model 3: CNN + Embeddings

In [42]:
keras.utils.set_random_seed(812)

filters = 64
kernel_size = 5

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embd_dim,
        input_length=max_length,
    ),
    Conv1D(filters, kernel_size=kernel_size, activation='relu'),
    GlobalAveragePooling1D(),
    Dense(6, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_compile(model)

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_5 (Embedding)     (None, 50, 20)            100000    
                                                                 
 conv1d_4 (Conv1D)           (None, 46, 64)            6464      
                                                                 
 global_average_pooling1d_2  (None, 64)                0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dense_12 (Dense)            (None, 6)                 390       
                                                                 
 dense_13 (Dense)            (None, 1)                 7         
                                                                 
Total params: 106861 (417.43 KB)
Trainable params: 106861 (417.43 KB)
Non-trainable params: 0 (0.00 Byte)
______________

In [43]:
%%time

model_fit(model)

Epoch 1/20
83/83 [==============================] - 1s 8ms/step - loss: 0.6731 - accuracy: 0.5553 - val_loss: 0.6336 - val_accuracy: 0.6643
Epoch 2/20
83/83 [==============================] - 0s 6ms/step - loss: 0.4970 - accuracy: 0.7860 - val_loss: 0.3949 - val_accuracy: 0.8394
Epoch 3/20
83/83 [==============================] - 0s 5ms/step - loss: 0.3085 - accuracy: 0.8830 - val_loss: 0.3578 - val_accuracy: 0.8553
Epoch 4/20
83/83 [==============================] - 0s 5ms/step - loss: 0.2370 - accuracy: 0.9148 - val_loss: 0.3633 - val_accuracy: 0.8538
Epoch 5/20
83/83 [==============================] - 0s 5ms/step - loss: 0.1969 - accuracy: 0.9329 - val_loss: 0.3794 - val_accuracy: 0.8577
Epoch 6/20
83/83 [==============================] - 0s 5ms/step - loss: 0.1657 - accuracy: 0.9445 - val_loss: 0.4177 - val_accuracy: 0.8447
Epoch 7/20
83/83 [==============================] - 0s 5ms/step - loss: 0.1422 - accuracy: 0.9563 - val_loss: 0.4537 - val_accuracy: 0.8413
Epoch 8/20
83/83 [==

## Model 4: LSTM + Embeddings

In [46]:
keras.utils.set_random_seed(812)

lstm_dim = 32

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embd_dim,
        input_length=max_length,
    ),
#     LSTM(lstm_dim),
    Bidirectional(LSTM(lstm_dim)),
    Dense(6, activation='relu'),
    Dense(1, activation='sigmoid')
])

model_compile(model)

Model: "sequential_9"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_7 (Embedding)     (None, 50, 20)            100000    
                                                                 
 bidirectional (Bidirection  (None, 64)                13568     
 al)                                                             
                                                                 
 dense_16 (Dense)            (None, 6)                 390       
                                                                 
 dense_17 (Dense)            (None, 1)                 7         
                                                                 
Total params: 113965 (445.18 KB)
Trainable params: 113965 (445.18 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [47]:
%%time

model_fit(model)

Epoch 1/20
83/83 [==============================] - 5s 29ms/step - loss: 0.6267 - accuracy: 0.6296 - val_loss: 0.5136 - val_accuracy: 0.7390
Epoch 2/20
83/83 [==============================] - 2s 19ms/step - loss: 0.3734 - accuracy: 0.8480 - val_loss: 0.3607 - val_accuracy: 0.8456
Epoch 3/20
83/83 [==============================] - 2s 19ms/step - loss: 0.2646 - accuracy: 0.8991 - val_loss: 0.3386 - val_accuracy: 0.8596
Epoch 4/20
83/83 [==============================] - 2s 20ms/step - loss: 0.2043 - accuracy: 0.9242 - val_loss: 0.3623 - val_accuracy: 0.8548
Epoch 5/20
83/83 [==============================] - 2s 20ms/step - loss: 0.1634 - accuracy: 0.9447 - val_loss: 0.4033 - val_accuracy: 0.8529
Epoch 6/20
83/83 [==============================] - 2s 20ms/step - loss: 0.1308 - accuracy: 0.9570 - val_loss: 0.4367 - val_accuracy: 0.8509
Epoch 7/20
83/83 [==============================] - 2s 19ms/step - loss: 0.1085 - accuracy: 0.9662 - val_loss: 0.5417 - val_accuracy: 0.8427
Epoch 8/20
83

# Mejora del modelo

In [50]:
df1 = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_sports.csv")
df2 = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_electronics.csv")
df3 = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/main/amazon_home.csv")
df = pd.concat([df1, df2, df3])

df['review_body'] = df['review_body'].str.replace('[^a-zA-ZñÑáéíóú .,]', '', regex=True)
df['review_body'] = df['review_body'].str.lower()
df = df[df.stars != 3]
df['good_product'] = (df.stars > 3).astype(int)
df.shape

(40329, 5)

In [51]:
X = df.review_body.values
y = df.good_product

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=33)

In [52]:
vocab_size = 5000

tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token='<OOV>'
)
tokenizer.fit_on_texts(X_train)

tokenized_train = tokenizer.texts_to_sequences(X_train)
tokenized_test = tokenizer.texts_to_sequences(X_test)

max_length = 50

padded_train = pad_sequences(tokenized_train, maxlen=max_length, truncating='post')
padded_test = pad_sequences(tokenized_test, maxlen=max_length, truncating='post')

In [53]:
print(len(padded_train))
print(len(padded_test))

32263
8066


In [54]:
# Reducción de overfitting

keras.utils.set_random_seed(812)

filters = 64
kernel_size = 5

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=embd_dim,
        input_length=max_length,
    ),
    Dropout(0.5),
    Conv1D(filters, kernel_size=kernel_size, activation='relu'),
    GlobalAveragePooling1D(),
    Dropout(0.5),
    Dense(6, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_compile(model)
model_fit(model)

Model: "sequential_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_9 (Embedding)     (None, 50, 20)            100000    
                                                                 
 dropout_3 (Dropout)         (None, 50, 20)            0         
                                                                 
 conv1d_6 (Conv1D)           (None, 46, 64)            6464      
                                                                 
 global_average_pooling1d_4  (None, 64)                0         
  (GlobalAveragePooling1D)                                       
                                                                 
 dropout_4 (Dropout)         (None, 64)                0         
                                                                 
 dense_20 (Dense)            (None, 6)                 390       
                                                     